# PART B: DATA ACQUISITION

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import sqlite3
import requests

## Load CSV File

In [ ]:
df_csv = pd.read_csv("customer_data.csv")
df_csv.head()

## Load JSON File

In [ ]:
import json

with open("customer_data.json") as f:
    data = json.load(f)

df_json = pd.DataFrame(data["customers"])
df_json.head()

## SQL Data

In [ ]:
conn = sqlite3.connect('customer_profiler.db')
pd.read_sql('SELECT * FROM orders LIMIT 5', conn)

## API Data

In [ ]:
url = "https://jsonplaceholder.typicode.com/users"

response = requests.get(url)
api_data = response.json()

df_api = pd.DataFrame(api_data)
df_api.head()

# PART C: DATA UNDERSTANDING & CLEANING

## Initial Exploration

In [ ]:
df = df_json.copy()   # use JSON dataset for analysis

df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## Missing Values & Duplicates

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

## Data Cleaning

In [ ]:
df['income'].fillna(df['income'].mean(), inplace=True)

print(df.isnull().sum())

## Convert Data Types

In [ ]:
conn = sqlite3.connect('customer_profiler.db')  # use sql dataset for analysis
df = pd.read_sql('SELECT * FROM orders', conn)

df.info()

In [ ]:
df['order_date'] = pd.to_datetime(df['order_date'])

df.info()

## Drop Irrelevant Columns

In [ ]:
df = df_json.copy()   # use JSON dataset for analysis
df.drop(columns=['gender'], inplace=True)


In [ ]:
df.head()

# PART D: EXPLORATORY DATA ANALYSIS (EDA)

## Univariate Analysis

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv('customer_data.csv')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Histogram
axes[0].hist(df['age'], bins=20, color="#eb2525", edgecolor='white', alpha=0.85)
axes[0].set_title('Age — Histogram')
axes[0].set_xlabel('Age'); axes[0].set_ylabel('Frequency')

# Box Plot
axes[2].boxplot(df['purchases'], patch_artist=True,
                boxprops=dict(facecolor='#cffafe', color='#0891b2'),
                medianprops=dict(color='#db2777', linewidth=2))
axes[2].set_title('purchases — Box Plot')

# KDE Plot
sns.kdeplot(df['income'], ax=axes[1], color="#862a72", fill=True, alpha=0.4)
axes[1].set_title('Income — KDE (Smooth Curve)')

plt.tight_layout()
plt.show()

print(f"Skewness: {df['age'].skew():.2f}")
print(f"Kurtosis: {df['age'].kurtosis():.2f}")


## Bivariate Analysis

In [ ]:
df = df_json.copy()

plt.figure(figsize=(7,5))

sns.regplot(
    data=df,
    x='income',
    y='churn',
    color="#C9392F",
    scatter_kws={'alpha': 0.4},
    line_kws={'color': "#aa8b24", 'linewidth': 2.5}
)

plt.xlabel('Income')
plt.ylabel('Churn')
plt.title('Income vs churn')
plt.show()



In [ ]:
df = df_csv.copy()

plt.figure(figsize=(8,5))

sns.violinplot(
    data=df,
    x='gender',
    y='purchases',
    palette='Set3',
    inner="box"
)

plt.title('Gender vs Purchases — Violin Plot')
plt.xticks(rotation=45)

plt.show()

In [ ]:
# Select only numerical columns
corr_matrix = df.select_dtypes(include='number').corr()

# Heatmap
plt.figure(figsize=(10, 8))

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    mask=mask,
    vmin=-1, vmax=1,
    linewidths=0.5,
    square=True,
    cbar_kws={'shrink': 0.8}
)

plt.title("Correlation Heatmap — Customer Data", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
sns.pairplot(
    df[['age', 'income', 'purchases', 'total_spent', 'churn']],
    hue='churn',
    diag_kind='kde',
    plot_kws={'alpha': 0.5}
)

plt.suptitle("Customer Data — Pair Plot by Churn", y=1.02)
plt.show()

In [ ]:
plt.figure(figsize=(10,6))

scatter = plt.scatter(
    df['income'],
    df['total_spent'],
    c=df['purchases'],
    s=df['membership_years'] * 50,
    cmap='viridis',
    alpha=0.6,
    edgecolors='white'
)

plt.colorbar(scatter, label='Purchases')
plt.xlabel("Income")
plt.ylabel("Total Spent")
plt.title("Multi-variable Relationship")

plt.show()

In [ ]:
!pip install ydata_profiling
import pandas as pd
from ydata_profiling import ProfileReport

# Load dataset
df = pd.read_json("customer_data.json")
df = pd.DataFrame(df["customers"])

# Generate full profiling report
profile = ProfileReport(
    df,
    title="Customer Data Profiling Report",
    explorative=True
)

# Save report as HTML
profile.to_file("customer_data_report.html")

print("Report generated successfully!")

# Display inside notebook
profile.to_notebook_iframe()

profile_minimal = ProfileReport(df, minimal=True)
profile_minimal.to_file("quick_report.html")